# BERT Tokenizer and Model Exploration

This notebook explores the main components of a BERT-based text classification pipeline:

- Loading a pre-trained tokenizer
- Tokenizing individual sentences and batches
- Inspecting input IDs and attention masks
- Loading a BERT model for four-class classification
- Examining logits and probabilities
- Creating and tokenizing a Hugging Face Dataset

> Note: The classification head has not yet been fine-tuned on the mental-health dataset. Therefore, the predictions produced in this notebook are only used to test the pipeline and are not meaningful model predictions.

In [36]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

In [37]:
MODEL_NAME = "bert-base-uncased"
CACHE_DIR = "../models/huggingface_cache"
NUM_LABELS = 4

id2label = {
    0: "Anxiety",
    1: "Depression",
    2: "Normal",
    3: "Suicidal",
}

label2id = {label: class_id for class_id, label in id2label.items()}

print("Label mapping:", id2label)

Label mapping: {0: 'Anxiety', 1: 'Depression', 2: 'Normal', 3: 'Suicidal'}


In [38]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
)

print("Tokenizer successfully loaded.")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model maximum length:", tokenizer.model_max_length)

Tokenizer successfully loaded.
Tokenizer class: BertTokenizer
Vocabulary size: 30522
Model maximum length: 512


In [39]:
sentence = "I feel very sad today."

single_encoded = tokenizer(
    sentence,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print("Original sentence:")
print(sentence)

print("\nInput IDs:")
print(single_encoded["input_ids"])

print("\nAttention mask:")
print(single_encoded["attention_mask"])

print("\nInput IDs shape:")
print(single_encoded["input_ids"].shape)

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(single_encoded["input_ids"][0]))

Original sentence:
I feel very sad today.

Input IDs:
tensor([[ 101, 1045, 2514, 2200, 6517, 2651, 1012,  102]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1]])

Input IDs shape:
torch.Size([1, 8])

Tokens:
['[CLS]', 'i', 'feel', 'very', 'sad', 'today', '.', '[SEP]']


In [42]:
sentences = [
    "I am happy.",
    "I feel sad.",
    "Today I feel very depressed and hopeless.",
]

batch_encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

print("Input IDs:")
print(batch_encoded["input_ids"])

print("\nAttention mask:")
print(batch_encoded["attention_mask"])

print("\nInput IDs shape:")
print(batch_encoded["input_ids"].shape)

print("\nAttention mask shape:")
print(batch_encoded["attention_mask"].shape)

Input IDs:
tensor([[  101,  1045,  2572,  3407,  1012,   102,     0,     0,     0,     0],
        [  101,  1045,  2514,  6517,  1012,   102,     0,     0,     0,     0],
        [  101,  2651,  1045,  2514,  2200, 14777,  1998, 20625,  1012,   102]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Input IDs shape:
torch.Size([3, 10])

Attention mask shape:
torch.Size([3, 10])


In [43]:
for index, token_ids in enumerate(batch_encoded["input_ids"]):
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print(f"Sentence {index + 1}:")
    print(sentences[index])
    print(tokens)
    print("-" * 60)

Sentence 1:
I am happy.
['[CLS]', 'i', 'am', 'happy', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
------------------------------------------------------------
Sentence 2:
I feel sad.
['[CLS]', 'i', 'feel', 'sad', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
------------------------------------------------------------
Sentence 3:
Today I feel very depressed and hopeless.
['[CLS]', 'today', 'i', 'feel', 'very', 'depressed', 'and', 'hopeless', '.', '[SEP]']
------------------------------------------------------------


In [44]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    cache_dir=CACHE_DIR,
)

print("BERT model successfully loaded.")
print("Number of labels:", model.config.num_labels)
print("Classifier layer:")
print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model successfully loaded.
Number of labels: 4
Classifier layer:
Linear(in_features=768, out_features=4, bias=True)


In [45]:
model.eval()

with torch.no_grad():
    outputs = model(**batch_encoded)

print(outputs)
print("\nLogits:")
print(outputs.logits)

print("\nLogits shape:")
print(outputs.logits.shape)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0615, -0.1757, -0.3514, -0.0432],
        [ 0.0049,  0.0706, -0.3168, -0.0335],
        [ 0.0466, -0.1363, -0.3625, -0.0490]]), hidden_states=None, attentions=None)

Logits:
tensor([[-0.0615, -0.1757, -0.3514, -0.0432],
        [ 0.0049,  0.0706, -0.3168, -0.0335],
        [ 0.0466, -0.1363, -0.3625, -0.0490]])

Logits shape:
torch.Size([3, 4])


In [46]:
probabilities = torch.softmax(
    outputs.logits,
    dim=1,
)

print("Probabilities:")
print(probabilities)

print("\nProbability sums:")
print(probabilities.sum(dim=1))

Probabilities:
tensor([[0.2733, 0.2438, 0.2045, 0.2784],
        [0.2663, 0.2844, 0.1930, 0.2563],
        [0.2936, 0.2445, 0.1950, 0.2668]])

Probability sums:
tensor([1., 1., 1.])


In [47]:
predicted_class_ids = torch.argmax(
    probabilities,
    dim=1,
)

predicted_labels = [id2label[class_id.item()] for class_id in predicted_class_ids]

print("Predicted class IDs:")
print(predicted_class_ids)

print("\nPredicted labels:")
print(predicted_labels)

Predicted class IDs:
tensor([3, 1, 0])

Predicted labels:
['Suicidal', 'Depression', 'Anxiety']


## Important note about the predictions

The BERT encoder contains pre-trained language representations, but the four-class classification head has not yet been trained on the mental-health dataset.

Therefore, the predictions below only demonstrate that the inference pipeline is functioning correctly. They should not be interpreted as valid mental-health classifications.

In [48]:
for sentence, class_id, probability_row in zip(
    sentences,
    predicted_class_ids,
    probabilities,
):
    class_id_value = class_id.item()
    predicted_label = id2label[class_id_value]
    confidence = probability_row[class_id_value].item()

    print("Text:", sentence)
    print("Prediction:", predicted_label)
    print("Confidence:", round(confidence, 4))
    print("-" * 60)

Text: I am happy.
Prediction: Suicidal
Confidence: 0.2784
------------------------------------------------------------
Text: I feel sad.
Prediction: Depression
Confidence: 0.2844
------------------------------------------------------------
Text: Today I feel very depressed and hopeless.
Prediction: Anxiety
Confidence: 0.2936
------------------------------------------------------------


In [49]:
sample_data = {
    "text": [
        "I am happy.",
        "I feel sad.",
        "Today I feel hopeless.",
    ],
    "label": [
        2,
        1,
        3,
    ],
}

sample_dataset = Dataset.from_dict(sample_data)

print(sample_dataset)
print("\nFirst example:")
print(sample_dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 3
})

First example:
{'text': 'I am happy.', 'label': 2}


In [50]:
def tokenize_function(batch):
    """Tokenize a batch of text examples."""
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )

In [51]:
tokenized_sample_dataset = sample_dataset.map(
    tokenize_function,
    batched=True,
)

print(tokenized_sample_dataset)

print("\nFirst tokenized example:")
print(tokenized_sample_dataset[0])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3
})

First tokenized example:
{'text': 'I am happy.', 'label': 2, 'input_ids': [101, 1045, 2572, 3407, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [52]:
first_example = tokenized_sample_dataset[0]

print("Text:")
print(first_example["text"])

print("\nLabel:")
print(first_example["label"])

print("\nInput IDs:")
print(first_example["input_ids"])

print("\nAttention mask:")
print(first_example["attention_mask"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(first_example["input_ids"]))

Text:
I am happy.

Label:
2

Input IDs:
[101, 1045, 2572, 3407, 1012, 102]

Attention mask:
[1, 1, 1, 1, 1, 1]

Tokens:
['[CLS]', 'i', 'am', 'happy', '.', '[SEP]']


In [ ]:
## Summary

In this notebook:

1. A `bert-base-uncased` tokenizer was loaded.
2. Individual sentences and batches were tokenized.
3. Input IDs, attention masks, padding, and tensor shapes were examined.
4. A BERT model with a four-class classification head was loaded.
5. Logits were converted into probabilities.
6. A small Hugging Face Dataset was created and tokenized.

The classification head has not yet been fine-tuned. The next stage of the project will use the real mental-health dataset to:

- clean and validate the labels,
- create training, validation, and test sets,
- fine-tune the BERT model,
- evaluate performance,
- and perform error analysis.